# Route profile features


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

In [2]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_data_dir = project_root / "data" / "processed"

train_df = pd.read_parquet(processed_data_dir / "train_with_folds.parquet")
test_df = pd.read_parquet(processed_data_dir / "test_clean.parquet")

print("train:", train_df.shape)
print("test:", test_df.shape)

train: (699635, 25)
test: (299844, 23)


## Fold split


In [3]:
valid_fold = 0
target = "satisfaction"
drop_columns = ["id", target, "fold"]

train_mask = train_df["fold"] != valid_fold
valid_mask = train_df["fold"] == valid_fold

x_train = train_df.loc[train_mask].drop(columns=drop_columns).copy()
x_valid = train_df.loc[valid_mask].drop(columns=drop_columns).copy()
x_test = test_df.drop(columns=["id"]).copy()

y_train = train_df.loc[train_mask, target]
y_valid = train_df.loc[valid_mask, target]

print("x_train:", x_train.shape)
print("x_valid:", x_valid.shape)
print("x_test:", x_test.shape)

x_train: (559708, 22)
x_valid: (139927, 22)
x_test: (299844, 22)


## Initial route features


In [4]:
route_column = "Flight Distance"
profile_column = "Online boarding"

route_counts = x_train[route_column].value_counts()
route_means = x_train.groupby(route_column)[profile_column].mean()
global_mean = x_train[profile_column].mean()

for frame in [x_train, x_valid, x_test]:
    frame["route_count"] = (
        frame[route_column].map(route_counts).fillna(0).astype("float32")
    )

    frame["route_online_boarding_mean"] = (
        frame[route_column].map(route_means).fillna(global_mean).astype("float32")
    )

    frame["route_online_boarding_residual"] = (
        frame[profile_column] - frame["route_online_boarding_mean"]
    ).astype("float32")

In [5]:
route_features = [
    "route_count",
    "route_online_boarding_mean",
    "route_online_boarding_residual",
]

print("new shapes:", x_train.shape, x_valid.shape, x_test.shape)
print("missing values:")
print(x_valid[route_features].isna().sum())

x_valid[[route_column, profile_column] + route_features].head()

new shapes: (559708, 25) (139927, 25) (299844, 25)
missing values:
route_count                       0
route_online_boarding_mean        0
route_online_boarding_residual    0
dtype: int64


,Flight Distance,Online boarding,route_count,route_online_boarding_mean,route_online_boarding_residual
2,1371,3,309.0,3.330097,-0.330097
9,588,3,3142.0,3.098027,-0.098027
11,2636,2,12.0,3.583333,-1.583333
13,946,2,2535.0,3.035897,-1.035897
17,631,2,2371.0,2.901729,-0.901729


## Full route profile


In [6]:
rating_columns = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Cleanliness",
]

profile_columns = rating_columns + [
    "Age",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes",
]

In [7]:
def clean_name(column):
    return (
        column.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("-", "_")
    )


def add_route_profiles(train_part, valid_part, test_part):
    train_part = train_part.copy()
    valid_part = valid_part.copy()
    test_part = test_part.copy()

    frames = [train_part, valid_part, test_part]

    arrival_column = "Arrival Delay in Minutes"
    arrival_median = train_part[arrival_column].median()

    for frame in frames:
        frame[arrival_column] = frame[arrival_column].fillna(arrival_median)

    route_counts = train_part[route_column].value_counts()
    route_stats = train_part.groupby(route_column)[profile_columns].agg(["mean", "std"])

    for frame in frames:
        frame["route_count"] = (
            frame[route_column].map(route_counts).fillna(0).astype("float32")
        )

        for column in profile_columns:
            name = clean_name(column)
            mean_name = f"route_{name}_mean"
            std_name = f"route_{name}_std"
            residual_name = f"route_{name}_residual"

            global_mean = train_part[column].mean()
            global_std = train_part[column].std()

            frame[mean_name] = (
                frame[route_column]
                .map(route_stats[(column, "mean")])
                .fillna(global_mean)
                .astype("float32")
            )

            frame[std_name] = (
                frame[route_column]
                .map(route_stats[(column, "std")])
                .fillna(global_std)
                .astype("float32")
            )

            frame[residual_name] = (
                frame[column] - frame[mean_name]
            ).astype("float32")

    return train_part, valid_part, test_part

In [8]:
x_train, x_valid, x_test = add_route_profiles(
    x_train,
    x_valid,
    x_test,
)

all_route_features = [
    column for column in x_train.columns if column.startswith("route_")
]

print("route features:", len(all_route_features))
print("new shapes:", x_train.shape, x_valid.shape, x_test.shape)
print("missing values in route features:", x_valid[all_route_features].isna().sum().sum())

route features: 49
new shapes: (559708, 71) (139927, 71) (299844, 71)
missing values in route features: 0


## Fold 0 validation


In [10]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

inner_folds = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

distance_encoder = TargetEncoder(
    target_type="binary",
    smooth="auto",
    cv=inner_folds,
)

x_train["flight_distance_te"] = distance_encoder.fit_transform(
    x_train[[route_column]],
    y_train,
).ravel()

x_valid["flight_distance_te"] = distance_encoder.transform(
    x_valid[[route_column]]
).ravel()

x_test["flight_distance_te"] = distance_encoder.transform(
    x_test[[route_column]]
).ravel()

In [11]:
class_categories = sorted(
    set(x_train["Class"].dropna().unique())
    | set(x_valid["Class"].dropna().unique())
    | set(x_test["Class"].dropna().unique())
)

for frame in [x_train, x_valid, x_test]:
    frame["Class"] = pd.Categorical(
        frame["Class"],
        categories=class_categories,
    )

print("features for model:", x_train.shape[1])

features for model: 72


In [12]:
parameters = {
    "n_estimators": 2000,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": 4,
    "verbosity": -1,
}

route_model = lgb.LGBMClassifier(**parameters)

route_model.fit(
    x_train,
    y_train,
    eval_set=[(x_valid, y_valid)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(100),
        lgb.log_evaluation(100),
    ],
)

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.955724	valid_0's binary_logloss: 0.247226
[200]	valid_0's auc: 0.958507	valid_0's binary_logloss: 0.232138
[300]	valid_0's auc: 0.959397	valid_0's binary_logloss: 0.229075
[400]	valid_0's auc: 0.959682	valid_0's binary_logloss: 0.228158
[500]	valid_0's auc: 0.959831	valid_0's binary_logloss: 0.227736
[600]	valid_0's auc: 0.959901	valid_0's binary_logloss: 0.227471
[700]	valid_0's auc: 0.959962	valid_0's binary_logloss: 0.227252
[800]	valid_0's auc: 0.960014	valid_0's binary_logloss: 0.227077
[900]	valid_0's auc: 0.960069	valid_0's binary_logloss: 0.226946
[1000]	valid_0's auc: 0.96008	valid_0's binary_logloss: 0.2269
[1100]	valid_0's auc: 0.960135	valid_0's binary_logloss: 0.226741
[1200]	valid_0's auc: 0.960136	valid_0's binary_logloss: 0.2267
[1300]	valid_0's auc: 0.960155	valid_0's binary_logloss: 0.226642
[1400]	valid_0's auc: 0.960182	valid_0's binary_logloss: 0.226566
[1500]	valid_0's auc: 0.9601

,learning_rate,0.03
,n_estimators,2000
,subsample,0.8
,subsample_freq,1
,colsample_bytree,0.8
,reg_lambda,1.0
,random_state,42
,n_jobs,4
,verbosity,-1
,boosting_type,'gbdt'
,num_leaves,31


In [13]:
valid_prediction = route_model.predict_proba(
    x_valid,
    num_iteration=route_model.best_iteration_,
)[:, 1]

route_profile_auc = roc_auc_score(y_valid, valid_prediction)
previous_fold_auc = 0.9603531995583718
difference = route_profile_auc - previous_fold_auc

print("previous fold 0:", previous_fold_auc)
print("route profile fold 0:", route_profile_auc)
print("difference:", difference)
print("best iteration:", route_model.best_iteration_)

previous fold 0: 0.9603531995583718
route profile fold 0: 0.9602160834737762
difference: -0.0001371160845956565
best iteration: 1577
